# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
tok = None
for env_candidate in ['.env', '../.env', '../../.env', '../../../.env']:
    if Path(env_candidate).exists():
        with open(env_candidate) as f:
            for line in f:
                if line.startswith('HF_TOKEN='):
                    tok = line.split('=',1)[1].strip()
                    print(f'Token loaded from {env_candidate}')
                    break
        if tok:
            break
if not tok:
    tok = os.environ.get('HF_TOKEN')
    if tok:
        print('Token loaded from environment')
    else:
        print('ERROR: HF_TOKEN not found in .env or environment')
con = duckdb.connect()
con.execute('SET enable_progress_bar = false')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{tok}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

Token loaded from ../../.env


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My verdict system
- CONFIRMED - The signal checks out as expected
- OPPOSITE - The signal is backwards (e.g., fresh when it should be stale)
- MIXED - Sometimes yes, sometimes no
- FALSE - This signal doesn't apply (e.g., no data)

### Two signals:
1. Staleness - How long has the page been since it was last updated? (Check days_since_last_update)
1. Low Click-Through Rate (CTR) - Are people clicking when they see this page in search results? (Check ctr and avg_position)

A page is worth reviewing if its staleness is long (high days_since_last_update) and it has low CTR.

### The scoring formula
`visibility_score` = a page's impressions compared to other pages
`freshness_risk_score` = how stale the page is (older = higher score)
`position_opportunity_score` = a page's potential score to rank higher in search

##### Adding weights to each of the score and compute the baseline_refresh_score
`baseline_refresh_score` = 0.4 * `visibility_score` + o.3 * `freshness_risk_score` + 0.25 * `position_opportunity_score` + 0.05 * `depth_gap_score`

### Reason Codes:
- `stale_visible_page` - Old page that still gets views
- `declining_with_demand` - Getting less traffic but people still want it
- `low_ctr_visible_page` - People see it but don't click
- `thin_visible_page` - Short content that gets views
- `low_engagement_visible_page` - People click but don't spend time reading

In [2]:
import pandas as pd

df = pd.read_csv("/Users/jasonpham/PycharmProjects/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows from starter dataset")

Loaded 30,000 rows from starter dataset


In [3]:
df

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.00,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.00,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.00,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.00,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.00,good,page_3_5,down,-34.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,content_c322796023c8,client_e29c9c180c,10.0,0.05,LOW,0.00,keyword article,transactional,1386.0,9084.0,...,8000-15000,0.00,0.0,0.00,0.00,0.00,low,top_3,new,NaN
29996,content_526572edb3fa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2654.0,17056.0,...,15000-25000,0.39,6.6,0.00,66.67,0.00,moderate,page_1,down,-75.1
29997,content_38112bdd0c6e,client_349c41201b,10.0,1.00,HIGH,0.00,keyword article,transactional,2857.0,18725.0,...,15000-25000,0.19,4.1,0.00,0.00,0.00,good,page_1,down,-66.2
29998,content_ab26273a7e7a,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,NaN,0.22,6.0,1.73,4.06,0.00,excellent,page_1,down,-27.9


In [4]:
print("Staleness:", len(df[df['days_since_last_update'] >= 180]))


Staleness: 174


In [5]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Signal 1: Staleness (freshness_tier)
print("--- Signal 1: Staleness ---")
bucket_1 = df.groupby('freshness_tier').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining_label', 'mean')
).sort_index()
print(bucket_1)
print("\nVerdict: MIXED - Staleness alone doesn't show a perfectly clean stair-step up in decline rate.")

# Signal 2: CTR vs Position
print("\n--- Signal 2: Low CTR on Page 1 ---")
page_1 = df[df['position_tier'] == 'page_1'].copy()
page_1['ctr_bucket'] = pd.qcut(page_1['ctr'], q=5, duplicates='drop')
bucket_2 = page_1.groupby('ctr_bucket').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining_label', 'mean')
).sort_index()
print(bucket_2)
print("\nVerdict: CONFIRMED - For a given position tier (page 1), lower CTR correlates with much higher decline rates.")

--- Signal 1: Staleness ---
                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
181+              174      0.471264
31-90             175      0.588571
91-180           9171      0.611057

Verdict: MIXED - Staleness alone doesn't show a perfectly clean stair-step up in decline rate.

--- Signal 2: Low CTR on Page 1 ---
                   n  decline_rate
ctr_bucket                        
(-0.001, 0.08]  4762      0.597228
(0.08, 0.23]    2341      0.604870
(0.23, 0.5]     2352      0.565051
(0.5, 100.0]    2359      0.483680

Verdict: CONFIRMED - For a given position tier (page 1), lower CTR correlates with much higher decline rates.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# Rule: We want to review pages that:
# 1. Have had some impressions (to make sure it's valid).
# 2. Have high staleness (days_since_last_update is high)
# 3. Have low CTR relative to others

import numpy as np

df['has_impressions'] = (df['impressions_90d'] > 10).astype(int)
df['visibility'] = np.log1p(df['impressions_90d'])
df['staleness_factor'] = np.where(df['days_since_last_update'].isna(), 0, np.clip(df['days_since_last_update'] / 365, 0, 1))

# Low CTR factor for page 1/2 results
df['low_ctr_factor'] = np.where(
    (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'].notna()),
    np.clip(1.0 - df['ctr'], 0, 1.0),
    0
)

df['baseline_refresh_score'] = df['has_impressions'] * df['visibility'] * df['staleness_factor'] * df['low_ctr_factor']

def assign_reason(row):
    if row['baseline_refresh_score'] == 0:
        return 'no_issue'
    if row['staleness_factor'] > 0.5 and row['low_ctr_factor'] > 0.8:
        return 'stale_and_low_ctr_visible'
    elif row['staleness_factor'] > 0.5:
        return 'stale_visible_page'
    elif row['low_ctr_factor'] > 0.8:
        return 'low_ctr_visible_page'
    return 'other'

df['reason_code'] = df.apply(assign_reason, axis=1)
df['action'] = np.where(df['baseline_refresh_score'] > 0, 'Review & Refresh', 'Leave as is')

queue = df[['content_id', 'client_id', 'baseline_refresh_score', 'reason_code', 'action']].sort_values('baseline_refresh_score', ascending=False)
import os
os.makedirs('../outputs', exist_ok=True)
queue.to_csv('../outputs/baseline_action_score.csv', index=False)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"Base rate (is_declining_label): {df['is_declining_label'].mean():.3f}")
print(f"Precision@50: {precision_at_k(df['baseline_refresh_score'], df['is_declining_label'], 50):.3f}")
print(f"Precision@100: {precision_at_k(df['baseline_refresh_score'], df['is_declining_label'], 100):.3f}")


Base rate (is_declining_label): 0.542
Precision@50: 0.660
Precision@100: 0.610


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top_20 = df.sort_values('baseline_refresh_score', ascending=False).head(20)

print("Top 20 Review:\n")
for i, (_, row) in enumerate(top_20.iterrows(), 1):
    action = row['action']
    reason = row['reason_code']
    confidence = "High" if row['baseline_refresh_score'] > 5 else "Medium"
    wrong_if = "If the page is purely navigational and users don't need to click, or if 'days_since_update' isn't reflecting actual content freshness."
    
    print(f"{i}. Action: {action} | Reason: {reason} | Conf: {confidence} | Why wrong? {wrong_if}")

print("\nLeakage Check: The score only uses impressions_90d, days_since_last_update, and avg_position/ctr. It strictly avoids trend_direction and trend_pct, so no forward leakage is possible.")


Top 20 Review:

1. Action: Review & Refresh | Reason: stale_and_low_ctr_visible | Conf: Medium | Why wrong? If the page is purely navigational and users don't need to click, or if 'days_since_update' isn't reflecting actual content freshness.
2. Action: Review & Refresh | Reason: stale_and_low_ctr_visible | Conf: Medium | Why wrong? If the page is purely navigational and users don't need to click, or if 'days_since_update' isn't reflecting actual content freshness.
3. Action: Review & Refresh | Reason: stale_and_low_ctr_visible | Conf: Medium | Why wrong? If the page is purely navigational and users don't need to click, or if 'days_since_update' isn't reflecting actual content freshness.
4. Action: Review & Refresh | Reason: stale_visible_page | Conf: Medium | Why wrong? If the page is purely navigational and users don't need to click, or if 'days_since_update' isn't reflecting actual content freshness.
5. Action: Review & Refresh | Reason: stale_and_low_ctr_visible | Conf: Medium | Wh

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
# No forward leakage check confirmed in Top-20 review code above.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.